# Feature Engineering

In [1]:
%run 00_functions.ipynb

In [2]:
df = pd.read_csv("../data/lstm/processed/bitcoin_cleaned_data.csv")

print(df.columns.tolist())
df.head()

['date', 'open', 'high', 'low', 'close', 'volume']


,date,open,high,low,close,volume
0,2018-01-01,13850.490234,13921.530273,12877.669922,13444.879883,78425
1,2018-01-02,13444.879883,15306.129883,12934.160156,14754.129883,137732
2,2018-01-03,14754.089844,15435.009766,14579.709961,15156.620117,106543
3,2018-01-04,15156.490234,15408.660156,14244.669922,15180.080078,110969
4,2018-01-05,15180.080078,17126.949219,14832.360352,16954.779297,141960


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3207 entries, 0 to 3206
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    3207 non-null   str    
 1   open    3207 non-null   float64
 2   high    3207 non-null   float64
 3   low     3207 non-null   float64
 4   close   3207 non-null   float64
 5   volume  3207 non-null   int64  
dtypes: float64(4), int64(1), str(1)
memory usage: 150.5 KB


### Relative Strength Index (RSI)

In [4]:
# Calculate daily price changes
price_change = df["close"].diff()

# Separate gains and losses
gain = price_change.clip(lower=0)
loss = -price_change.clip(upper=0)

# Calculate average gains and losses
avg_gain = gain.ewm(
    alpha=1/14,
    min_periods=14,
    adjust=False
).mean()

avg_loss = loss.ewm(
    alpha=1/14,
    min_periods=14,
    adjust=False
).mean()

# Calculate RSI
rs = avg_gain / avg_loss
df["RSI"] = 100 - (100 / (1 + rs))

In [5]:
df[["date", "close", "RSI"]].head(20)

,date,close,RSI
0,2018-01-01,13444.879883,NaN
1,2018-01-02,14754.129883,NaN
2,2018-01-03,15156.620117,NaN
3,2018-01-04,15180.080078,NaN
4,2018-01-05,16954.779297,NaN
5,2018-01-06,17172.300781,NaN
6,2018-01-07,16228.160156,NaN
7,2018-01-08,14976.169922,NaN
8,2018-01-09,14778.496094,NaN
9,2018-01-10,15042.963867,NaN


### Simple Moving Average (SMA)

In [6]:
# Calculate the 20-day simple moving average
df["SMA"] = df["close"].rolling(window=20).mean()

In [7]:
df[["date", "close", "SMA"]].head(25)

,date,close,SMA
0,2018-01-01,13444.879883,NaN
1,2018-01-02,14754.129883,NaN
2,2018-01-03,15156.620117,NaN
3,2018-01-04,15180.080078,NaN
4,2018-01-05,16954.779297,NaN
5,2018-01-06,17172.300781,NaN
6,2018-01-07,16228.160156,NaN
7,2018-01-08,14976.169922,NaN
8,2018-01-09,14778.496094,NaN
9,2018-01-10,15042.963867,NaN


### Moving Average Convergence Divergence (MACD)

In [8]:
# Calculate the 12-day and 26-day exponential moving averages
ema_12 = df["close"].ewm(span=12, adjust=False).mean()
ema_26 = df["close"].ewm(span=26, adjust=False).mean()

# Calculate MACD
df["MACD"] = ema_12 - ema_26

In [9]:
df[["date", "close", "MACD"]].head(35)

,date,close,MACD
0,2018-01-01,13444.879883,0.000000
1,2018-01-02,14754.129883,104.441595
2,2018-01-03,15156.620117,217.186322
3,2018-01-04,15180.080078,304.915477
4,2018-01-05,16954.779297,511.745772
5,2018-01-06,17172.300781,685.312581
6,2018-01-07,16228.160156,738.172081
7,2018-01-08,14976.169922,671.300137
8,2018-01-09,14778.496094,595.488592
9,2018-01-10,15042.963867,550.403031


In [10]:
# Check the available columns
df.columns.tolist()

['date', 'open', 'high', 'low', 'close', 'volume', 'RSI', 'SMA', 'MACD']

In [11]:
# Check missing values in the engineered features
df[["MACD", "RSI", "SMA"]].isna().sum()

MACD     0
RSI     14
SMA     19
dtype: int64

In [12]:
# Remove rows with missing engineered features
df = df.dropna(subset=["MACD", "RSI", "SMA"]).reset_index(drop=True)

# Check the resulting dataset
print("Dataset shape:", df.shape)
print("Missing values in engineered features:")
print(df[["MACD", "RSI", "SMA"]].isna().sum())

df.head()

Dataset shape: (3188, 9)
Missing values in engineered features:
MACD    0
RSI     0
SMA     0
dtype: int64


,date,open,high,low,close,volume,RSI,SMA,MACD
0,2018-01-20,11579.240234,13082.238281,11567.291016,12858.858398,143900,66.179585,14051.718457,-569.638525
1,2018-01-21,12857.081055,12860.202148,11164.538086,11594.896484,206959,59.335267,13959.219287,-630.799598
2,2018-01-22,11597.452148,11970.898438,10136.767578,10864.833984,201726,55.748913,13764.754492,-729.767823
3,2018-01-23,10869.032227,11437.955078,10056.577148,10903.190430,188365,55.899728,13552.083008,-795.930814
4,2018-01-24,10899.393555,11549.520508,10584.529297,11452.302734,136064,58.101280,13365.694141,-794.893648


In [13]:
# Save the feature-engineered dataset
df.to_csv(
    "../data/lstm/processed/bitcoin_feature_engineered.csv",
    index=False
)

print("Feature-engineered dataset saved successfully.")

Feature-engineered dataset saved successfully.


In [14]:
print("First date:", df["date"].iloc[0])
print("Last date:", df["date"].iloc[-1])
print("Number of rows:", len(df))
print("Columns:", df.columns.tolist())

First date: 2018-01-20
Last date: 2026-10-12
Number of rows: 3188
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'RSI', 'SMA', 'MACD']
